In [ ]:
%%capture
using_colab = True

try:
    import google.colab  # type: ignore
    using_colab = True
except ImportError:
    using_colab = False

if using_colab:
    %pip install  "gpjax==0.11.2" optax


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np

# addtional JAX libs
import gpjax as gpx
import optax

# graphics stuff
import matplotlib.pyplot as plt
import plotly.graph_objects as go


# for rendering in vscdoe 
if using_colab:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

In [ ]:
# ------------------------------------------------------------
# Ground-truth latent-space dummy function
# ------------------------------------------------------------
def latent_function(x: jnp.ndarray) -> jnp.ndarray:
    return jnp.sin(x)+ jnp.cos(2*x)

# ------------------------------------------------------------
# Fit GP model
# ------------------------------------------------------------
def fit_gp(X: jnp.ndarray, y: jnp.ndarray, key: jax.Array):
    # ARD-ready kernel; for 1D this is just a scalar lengthscale
    kernel = gpx.kernels.Matern52()
    meanf = gpx.mean_functions.Zero()

    prior = gpx.gps.Prior(mean_function=meanf, kernel=kernel)
    likelihood = gpx.likelihoods.Gaussian(num_datapoints=X.shape[0])
    posterior = prior * likelihood

    D = gpx.Dataset(X=X, y=y)

    objective = lambda model, data: -gpx.objectives.conjugate_mll(model, data)
    opt = optax.adam(1e-2)

    posterior, history = gpx.fit(
        model=posterior,
        objective=objective,
        train_data=D,
        optim=opt,
        num_iters=500,
        key=key,
    )

    return posterior, D, history

# ------------------------------------------------------------
# Predictive mean and variance
# ------------------------------------------------------------
def _maybe_call(x):
    return x() if callable(x) else x

def predict_gp(posterior, D, Xtest: jnp.ndarray):
    latent_dist = posterior.predict(Xtest, train_data=D)
    predictive_dist = posterior.likelihood(latent_dist)

    mu = _maybe_call(predictive_dist.mean).reshape(-1)
    var = _maybe_call(predictive_dist.variance).reshape(-1)
    return mu, var

# ------------------------------------------------------------
# Acquisition: uncertainty sampling
# x_next = argmax variance(x)
# ------------------------------------------------------------
def select_next_by_uncertainty(
    posterior,
    D,
    Xcand: jnp.ndarray,
    Xtrain: jnp.ndarray,
    min_dist: float = 1e-3,
):
    _, var = predict_gp(posterior, D, Xcand)

    # avoid sampling points already present
    dists = jnp.min(jnp.abs(Xcand[:, None, :] - Xtrain[None, :, :]), axis=(1, 2))
    masked_var = jnp.where(dists < min_dist, -jnp.inf, var)

    idx = jnp.argmax(masked_var)
    return Xcand[idx:idx+1], var, idx

# ------------------------------------------------------------
# Sequential uncertainty reduction loop
# ------------------------------------------------------------
def run_uncertainty_reduction(
    n_init: int = 5,
    n_iter: int = 8,
    x_min: float = 0.0,
    x_max: float = 2.0 * jnp.pi,
    seed: int = 0,
):
    key = jax.random.PRNGKey(seed)

    # Initial training data
    X = jnp.linspace(x_min, x_max, n_init).reshape(-1, 1)
    y = latent_function(X).reshape(-1, 1)

    # Dense candidate grid
    Xcand = jnp.linspace(x_min, x_max, 400).reshape(-1, 1)

    history = []

    for i in range(n_iter):
        key, subkey = jax.random.split(key)

        # GP on current dataset
        posterior, D, _ = fit_gp(X, y, subkey)
        mu, var = predict_gp(posterior, D, Xcand)

        # Select next point from current uncertainty
        Xnext, cand_var, idx = select_next_by_uncertainty(
            posterior=posterior,
            D=D,
            Xcand=Xcand,
            Xtrain=X,
        )

        ynext = latent_function(Xnext).reshape(-1, 1)

        # Augment dataset
        X = jnp.concatenate([X, Xnext], axis=0)
        y = jnp.concatenate([y, ynext], axis=0)

        history.append(
            {
                "iter": i + 1,
                "stage": "acquisition",   # GP before adding Xnext
                "X": X,
                "y": y,
                "Xcand": Xcand,
                "mu": mu,
                "var": var,
                "Xnext": Xnext,
                "ynext": ynext,
                "idx_next": idx,
            }
        )

        print(
            f"iter {i+1:02d}: "
            f"x_next = {float(Xnext[0,0]):.4f}, "
            f"std = {float(jnp.sqrt(cand_var[idx])):.4f}, "
            f"y_next = {float(ynext[0,0]):.4f}"
        )

        # ------------------------------------------------------------
        # Final GP fit after last acquired point has been added
        # ------------------------------------------------------------
        key, subkey = jax.random.split(key)
        posterior_final, D_final, _ = fit_gp(X, y, subkey)
        mu_final, var_final = predict_gp(posterior_final, D_final, Xcand)

        history.append(
            {
                "iter": n_iter,
                "stage": "final",         # GP after last point has been added
                "X": X,
                "y": y,
                "Xcand": Xcand,
                "mu": mu_final,
                "var": var_final,
                "Xnext": X[-1:],
                "ynext": y[-1:],
                "idx_next": None,
                "posterior": posterior_final,
                "D": D_final,
            }
        )

    return X, y, Xcand, history

In [ ]:
X, y, Xcand, history = run_uncertainty_reduction(
    n_init=5,
    n_iter=8,
    x_min=0.0,
    x_max=2.0 * jnp.pi,
    seed=42,
)

In [ ]:
def make_uncertainty_animation(
    Xcand,
    history,
    title="Sequential uncertainty reduction of dummy latent space function",
):
    x_grid = np.asarray(Xcand[:, 0])
    y_true = np.asarray(latent_function(Xcand).reshape(-1))

    frames = []
    slider_steps = []

    for k, h in enumerate(history):
        Xk = np.asarray(h["X"][:, 0])
        yk = np.asarray(h["y"][:, 0])
        mu = np.asarray(h["mu"]).reshape(-1)
        var = np.asarray(h["var"]).reshape(-1)
        std = np.sqrt(np.maximum(var, 0.0))

        upper = mu + 2.0 * std
        lower = mu - 2.0 * std

        stage = h.get("stage", "acquisition")
        iter_label = h.get("iter", k + 1)

        if stage == "acquisition":
            x_next = float(np.asarray(h["Xnext"])[0, 0])
            y_next = float(np.asarray(h["ynext"])[0, 0])

            candidate_trace = go.Scatter(
                x=[x_next],
                y=[y_next],
                mode="markers",
                name="new candidate",
                marker=dict(size=14, symbol="x"),
            )
            subtitle = f"Iteration {iter_label} — acquisition"
        else:
            # final GP after last sample has been added
            x_last = float(Xk[-1])
            y_last = float(yk[-1])

            candidate_trace = go.Scatter(
                x=[x_last],
                y=[y_last],
                mode="markers",
                name="last added point",
                marker=dict(size=14, symbol="circle-open"),
            )
            subtitle = f"Iteration {iter_label} — final refit"

        frame = go.Frame(
            name=str(k),
            data=[
                go.Scatter(
                    x=x_grid,
                    y=y_true,
                    mode="lines",
                    name="true function",
                    line=dict(width=2),
                ),
                go.Scatter(
                    x=x_grid,
                    y=mu,
                    mode="lines",
                    name="GP mean",
                    line=dict(width=2),
                ),
                go.Scatter(
                    x=x_grid,
                    y=upper,
                    mode="lines",
                    line=dict(width=0),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                go.Scatter(
                    x=x_grid,
                    y=lower,
                    mode="lines",
                    line=dict(width=0),
                    fill="tonexty",
                    name="GP ± 2σ",
                    hoverinfo="skip",
                ),
                go.Scatter(
                    x=Xk,
                    y=yk,
                    mode="markers",
                    name="sampled points",
                    marker=dict(size=9),
                ),
                candidate_trace,
            ],
            layout=go.Layout(
                title=f"{title}<br><sup>{subtitle}</sup>"
            ),
        )
        frames.append(frame)

        slider_steps.append(
            {
                "args": [
                    [str(k)],
                    {
                        "frame": {"duration": 400, "redraw": True},
                        "mode": "immediate",
                        "transition": {"duration": 500},
                    },
                ],
                "label": subtitle,
                "method": "animate",
            }
        )

    # initialize from first frame
    first = history[0]
    mu0 = np.asarray(first["mu"]).reshape(-1)
    var0 = np.asarray(first["var"]).reshape(-1)
    std0 = np.sqrt(np.maximum(var0, 0.0))
    X0 = np.asarray(first["X"][:, 0])
    y0 = np.asarray(first["y"][:, 0])

    first_stage = first.get("stage", "acquisition")
    first_iter = first.get("iter", 1)

    if first_stage == "acquisition":
        x_mark0 = float(np.asarray(first["Xnext"])[0, 0])
        y_mark0 = float(np.asarray(first["ynext"])[0, 0])
        mark_name0 = "new candidate"
        mark_symbol0 = "x"
        subtitle0 = f"Iteration {first_iter} — acquisition"
    else:
        x_mark0 = float(X0[-1])
        y_mark0 = float(y0[-1])
        mark_name0 = "last added point"
        mark_symbol0 = "circle-open"
        subtitle0 = f"Iteration {first_iter} — final refit"

    fig = go.Figure(
        data=[
            go.Scatter(
                x=x_grid,
                y=y_true,
                mode="lines",
                name="true function",
                line=dict(width=2),
            ),
            go.Scatter(
                x=x_grid,
                y=mu0,
                mode="lines",
                name="GP mean",
                line=dict(width=2),
            ),
            go.Scatter(
                x=x_grid,
                y=mu0 + 2.0 * std0,
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ),
            go.Scatter(
                x=x_grid,
                y=mu0 - 2.0 * std0,
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                name="GP ± 2σ",
                hoverinfo="skip",
            ),
            go.Scatter(
                x=X0,
                y=y0,
                mode="markers",
                name="sampled points",
                marker=dict(size=9),
            ),
            go.Scatter(
                x=[x_mark0],
                y=[y_mark0],
                mode="markers",
                name=mark_name0,
                marker=dict(size=14, symbol=mark_symbol0),
            ),
        ],
        frames=frames,
    )

    fig.update_layout(
        title=f"{title}<br><sup>{subtitle0}</sup>",
        xaxis_title="latent variable x",
        yaxis_title="f(x)",
        template="plotly_white",
        hovermode="x unified",
        sliders=[
            {
                "active": 0,
                "pad": {"t": 40},
                "steps": slider_steps,
                "currentvalue": {"prefix": "Frame: "},
            }
        ],
        updatemenus=[
            {
                "type": "buttons",
                "showactive": False,
                "x": 1.02,
                "y": 1.15,
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [
                            None,
                            {
                                "frame": {"duration": 500, "redraw": True},
                                "fromcurrent": True,
                                "transition": {"duration": 250},
                            },
                        ],
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {
                                "frame": {"duration": 0, "redraw": False},
                                "mode": "immediate",
                                "transition": {"duration": 0},
                            },
                        ],
                    },
                ],
            }
        ],
    )

    return fig

In [ ]:
fig = make_uncertainty_animation(Xcand, history)
fig.show()